# DriveSense-VLM — 02: Model Optimization

**GPU**: A100 80GB (required) | **Time**: ~1.5 h | **Cost**: ~18 CU

Three-stage pipeline (each stage idempotent internally but always invoked):
1. **LoRA merge** — fold adapter into base model (bfloat16)
2. **AWQ 4-bit quantization** — LLM decoder only; ViT stays fp16
3. **TensorRT ViT compilation** — falls back to `torch.compile`

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **Prerequisites**: `01_training.ipynb` must have saved a checkpoint.

In [1]:
# ══════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════
RUN_MOCK = False   # Set True to test the pipeline without a real checkpoint
# ══════════════════════════════════════════════════════════

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD (skip if already cloned — saves bandwidth)
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)
print(f"Working dir: {os.getcwd()}")

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

Mounted at /content/drive
Cloning into '/content/DriveSense-VLM'...
remote: Enumerating objects: 292, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 292 (delta 131), reused 246 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (292/292), 449.01 KiB | 6.80 MiB/s, done.
Resolving deltas: 100% (131/131), done.
Working dir: /content/DriveSense-VLM
✓ Repo:  /content/DriveSense-VLM
✓ Drive: /content/drive/MyDrive/DriveSense-VLM


In [2]:
# Install optimization dependencies
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate -q 2>&1 | tail -1
!pip install autoawq -q 2>&1 | tail -1
!pip install tensorrt -q 2>&1 | tail -1 || echo "TensorRT not available — torch.compile fallback"

import torch
assert torch.cuda.is_available(), "No GPU! Set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import drivesense
print("✓ DriveSense loaded")

ipython 7.34.0 requires jedi>=0.16, which is not installed.
✓ GPU : NVIDIA A100-SXM4-40GB
✓ VRAM: 42.4 GB
✓ DriveSense loaded


In [7]:
!pip install "torchao>=0.16.0" -q 2>&1 | tail -1

In [25]:
!pip install -U bitsandbytes>=0.46.1

In [26]:
os.chdir(REPO_ROOT)

# Upgrade back to transformers 5.0
!pip install transformers==5.0.0 -q 2>&1 | tail -1

# Quantize with bitsandbytes (built into transformers, no patches needed)
with open('/tmp/run_bnb_quant.py', 'w') as f:
    f.write("""
import sys, os, json, time
sys.path.insert(0, 'src')

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

merged_dir = "outputs/merged_model"
output_dir = "outputs/quantized_model"
os.makedirs(output_dir, exist_ok=True)

print("Loading merged model with 4-bit quantization...")
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    merged_dir,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(merged_dir)

load_time = time.time() - t0
print(f"Loaded in {load_time:.1f}s")

# Measure VRAM
vram_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used: {vram_gb:.2f} GB")

# Save quantized model
print("Saving quantized model...")
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

# Save quant config for reference
quant_info = {
    "method": "bitsandbytes",
    "bits": 4,
    "quant_type": "nf4",
    "double_quant": True,
    "compute_dtype": "bfloat16",
    "vram_gb": round(vram_gb, 2),
    "load_time_s": round(load_time, 1),
}
with open(f"{output_dir}/quant_config.json", "w") as f:
    json.dump(quant_info, f, indent=2)

# Size comparison
merged_size = sum(os.path.getsize(f"{merged_dir}/{f}") for f in os.listdir(merged_dir) if f.endswith('.safetensors')) / 1e9
quant_size = sum(os.path.getsize(f"{output_dir}/{f}") for f in os.listdir(output_dir) if f.endswith('.safetensors')) / 1e9

print(f"\\nMerged model: {merged_size:.2f} GB")
print(f"Quantized:    {quant_size:.2f} GB")
print(f"Compression:  {merged_size/max(quant_size, 0.01):.1f}x")
print(f"\\n✅ 4-bit quantized model saved to {output_dir}")
""")

!cd {REPO_ROOT} && python /tmp/run_bnb_quant.py 2>&1

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading merged model with 4-bit quantization...
`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_interleaved'}
Loading weights: 100% 632/632 [00:03<00:00, 192.93it/s, Materializing param=model.visual.patch_embed.proj.weight]
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_interleaved'}
Loaded in 10.4s
VRAM used: 1.58 GB
Saving quantized model...
Writing model shards: 100% 1/1 [00:03<00:00,  3.88s/it]

Merged model: 4.30 GB
Quantized:    1.58 GB
Compression:  2.7x

✅ 4-bit quantized model saved to outputs/quantized_model


In [32]:
os.chdir(REPO_ROOT)

with open('/tmp/run_compile_v2.py', 'w') as f:
    f.write("""
import sys, os, json, time
sys.path.insert(0, 'src')

import torch
torch.set_float32_matmul_precision('high')

from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from PIL import Image
import glob

output_dir = "outputs/tensorrt"
os.makedirs(output_dir, exist_ok=True)

print("Loading 4-bit model...")
model = AutoModelForImageTextToText.from_pretrained(
    "outputs/quantized_model", device_map="auto", dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained("outputs/merged_model")

# Test image
images = glob.glob("outputs/data/nuscenes_filtered/images/*.jpg")
img = Image.open(images[0]) if images else Image.new("RGB", (672, 448))

messages = [{"role": "user", "content": [
    {"type": "image", "image": img},
    {"type": "text", "text": "Describe this image briefly."}
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")

# Eager baseline (already measured: ~3400ms)
print("\\nEager warmup...")
for _ in range(2):
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=50)

times = []
for _ in range(5):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=50)
    torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)
eager_ms = sum(times) / len(times)
print(f"Eager: {eager_ms:.0f} ms")

# torch.compile with mode="default" (no CUDA graphs)
print("\\ntorch.compile (mode=default)...")
try:
    model.forward = torch.compile(model.forward, mode="default")
    print("  Compiling (first runs are slow)...")
    for _ in range(3):
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=50)

    times = []
    for _ in range(5):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=50)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    compiled_ms = sum(times) / len(times)
    speedup = eager_ms / compiled_ms
    method = "torch.compile(mode=default)"
    print(f"  Compiled: {compiled_ms:.0f} ms | Speedup: {speedup:.2f}x")
except Exception as e:
    print(f"  Failed: {e}")
    compiled_ms = eager_ms
    speedup = 1.0
    method = "eager_only"

results = {
    "method": method,
    "eager_mean_ms": round(eager_ms),
    "compiled_mean_ms": round(compiled_ms),
    "speedup": round(speedup, 2),
    "max_new_tokens": 50,
    "quantization": "nf4_4bit",
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.memory_allocated()/1e9, 2),
    "vram_peak_gb": round(torch.cuda.max_memory_allocated()/1e9, 2),
}
with open(f"{output_dir}/fallback_info.json", "w") as f:
    json.dump(results, f, indent=2)
with open(f"{output_dir}/optimization_report.txt", "w") as f:
    for k, v in results.items():
        f.write(f"{k}: {v}\\n")

print(f"\\n{'='*50}")
for k, v in results.items():
    print(f"  {k}: {v}")
print(f"{'='*50}")
""")

!cd {REPO_ROOT} && python /tmp/run_compile_v2.py 2>&1

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading 4-bit model...
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_interleaved'}
Loading weights: 100% 632/632 [00:01<00:00, 494.05it/s, Materializing param=model.visual.patch_embed.proj.weight]
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_interleaved'}

Eager warmup...
Eager: 3695 ms

torch.compile (mode=default)...
  Compiling (first runs are slow)...
  Compiled: 2092 ms | Speedup: 1.77x

  method: torch.compile(mode=default)
  eager_mean_ms: 3695
  compiled_mean_ms: 2092
  speedup: 1.77
  max_new_tokens: 50
  quantization: nf4_4bit
  gpu: NVIDIA A100-SXM4-40GB
  vram_gb: 1.61
  vram_peak_gb: 1.85


In [35]:
# Final verification
from pathlib import Path
import json

print("=" * 60)
print("  DriveSense-VLM Optimization — Final Results")
print("=" * 60)

for label, path, key_file in [
    ("LoRA Merge",       "outputs/merged_model",     "model.safetensors"),
    ("NF4 4-bit Quant",  "outputs/quantized_model",  "quant_config.json"),
    ("torch.compile",    "outputs/tensorrt",          "fallback_info.json"),
]:
    full = f"{path}/{key_file}"
    if os.path.exists(full):
        size = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9
        print(f"  ✓ {label:20s}: {size:.2f} GB")
        if key_file in ("quant_config.json", "fallback_info.json"):
            cfg = json.load(open(full))
            for k, v in cfg.items():
                print(f"      {k}: {v}")
    else:
        print(f"  ✗ {label:20s}: not found")

  DriveSense-VLM Optimization — Final Results
  ✓ LoRA Merge          : 4.32 GB
  ✓ NF4 4-bit Quant     : 1.59 GB
      method: bitsandbytes
      bits: 4
      quant_type: nf4
      double_quant: True
      compute_dtype: bfloat16
      vram_gb: 1.58
      load_time_s: 10.4
  ✓ torch.compile       : 0.00 GB
      method: torch.compile(mode=default)
      eager_mean_ms: 3695
      compiled_mean_ms: 2092
      speedup: 1.77
      max_new_tokens: 50
      quantization: nf4_4bit
      gpu: NVIDIA A100-SXM4-40GB
      vram_gb: 1.61
      vram_peak_gb: 1.85
